In [ ]:
# =========================================================================
# CELL 1: KẾT NỐI GOOGLE DRIVE & ĐỒNG BỘ CODE TỪ GITHUB (DEVELOP BRANCH)
# =========================================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Cấu hình URL repository của bạn
REPO_URL = "https://github.com/CodeDaoVietNam/LIVR-Mini-Benchmark.git"
PROJECT_DIR = "LIVR-Mini-Benchmark"
BRANCH = "develop"

%cd /content
import os
if not os.path.exists(PROJECT_DIR):
    print(f"---> Đang thực hiện clone repo {REPO_URL} (nhánh {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
    %cd {PROJECT_DIR}
else:
    print(f"---> Repo {PROJECT_DIR} đã tồn tại. Đang tiến hành pull code mới nhất từ nhánh {BRANCH}...")
    %cd {PROJECT_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

In [ ]:
# =========================================================================
# CELL 2: CÀI ĐẶT THƯ VIỆN & CHẠY PIPELINE TIỀN XỬ LÝ DỮ LIỆU (TUẦN 1 & TUẦN 2)
# =========================================================================
# Cài đặt các thư viện cần thiết trên môi trường Colab
!pip install -r requirements.txt

import sys
import os
# Đảm bảo Python nhận diện được module trong thư mục src/
sys.path.append(os.getcwd())

# 1. Nạp và kiểm tra dữ liệu gốc (Nghiệm thu Tuần 1)
from src.utils import load_and_inspect_livr_dataset, filter_and_deduplicate_pipeline
dataset = load_and_inspect_livr_dataset()

# 2. Chạy thử nghiệm bộ tiền xử lý và khử trùng lặp trực quan (Nghiệm thu Tuần 2)
cleaned_dataset = filter_and_deduplicate_pipeline(dataset)

In [ ]:
# =========================================================================
# CELL 3: ĐỌC CONFIG, KHỞI TẠO MÔ HÌNH VÀ BỘ TỐI ƯU (TUẦN 4 - BƯỚC 1)
# =========================================================================
import json
import torch
from transformers import AdamW
from src.model import LIVRModelManager
from src.mask import patch_model_for_livr

# 1. Đọc file cấu hình định nghĩa sẵn
with open("config/implement_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)
print("IMPLEMENTATION CONFIGURATION:")
print(json.dumps(config, indent=2))

# 2. Dựng mô hình LIVR cấu hình K từ file config
manager = LIVRModelManager(model_id=config["model_id"], K=config["K"], device="cuda")
model = manager.setup_peft_and_freezing(
    r=config["lora_r"],
    alpha=config["lora_alpha"],
    dropout=config["lora_dropout"]
)
processor = manager.processor

# Monkey-patch model với Custom Attention Mask
patch_model_for_livr(
    model=model,
    latent_token_ids=manager.latent_token_ids,
    image_pad_token_id=manager.image_pad_token_id,
    pad_token_id=manager.pad_token_id
)

# 3. Khai báo bộ tối ưu AdamW sử dụng cấu hình từ file config
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=config["stage1_lr"], weight_decay=config["weight_decay"])

print("-> Đã khởi tạo cấu hình model và Optimizer thành công.")

In [ ]:
# =========================================================================
# CELL 4: VÒNG LẶP HUẤN LUYỆN CHÍNH (TRAINING LOOP - TUẦN 4 - BƯỚC 2)
# =========================================================================
import os
from tqdm import tqdm
from src.utils import prepare_vqa_inputs

# Lấy các tham số huấn luyện trực tiếp từ file config
STAGE1_EPOCHS = config["stage1_epochs"]
STAGE2_EPOCHS = config["stage2_epochs"]
TOTAL_EPOCHS = STAGE1_EPOCHS + STAGE2_EPOCHS
GRADIENT_ACCUMULATION_STEPS = config["grad_accumulation_steps"]
STAGE2_LR = config["stage2_lr"]

model.train()

print("====== CHÍNH THỨC KHỞI ĐỘNG VÒNG LẶP HUẤN LUYỆN 2 GIAI ĐOẠN ======")

for epoch in range(1, TOTAL_EPOCHS + 1):
    # Xác định Giai đoạn hiện tại để điều khiển mặt nạ mã nguồn
    current_stage = 1 if epoch <= STAGE1_EPOCHS else 2
    model.livr_stage = current_stage
    
    # Reset hoặc giảm Learning Rate khi chuyển sang Stage 2 theo đúng paper
    if epoch == STAGE1_EPOCHS + 1:
        print(f"\n➔ CHUYỂN GIAI ĐOẠN: Hạ Learning Rate xuống {STAGE2_LR} cho Stage 2...")
        for param_group in optimizer.param_groups:
            param_group['lr'] = STAGE2_LR
            
    # Khởi tạo loss và reset gradients cho mỗi epoch ở cấp độ vòng lặp epoch
    epoch_loss = 0.0
    optimizer.zero_grad()
    
    # Thanh tiến trình theo dõi tiến độ từng epoch
    progress_bar = tqdm(cleaned_dataset, desc=f"Epoch {epoch}/{TOTAL_EPOCHS} (Stage {current_stage})")
    
    for step, batch in enumerate(progress_bar):
        # Sử dụng hàm prepare_vqa_inputs để chuẩn bị Tensor đầu vào đạt chuẩn cho Qwen
        inputs = prepare_vqa_inputs(
            processor=processor,
            conversation=batch['conversation'],
            latent_tokens=manager.latent_tokens,
            device="cuda"
        )
        
        # --- KỸ THUẬT CAN THIỆP PHÂN PHỐI MẶT NẠ CHÚ Ý (STAGING) ---
        # Hàm forward của model đã được chúng ta đè (Monkey-patched) từ tuần trước.
        # Hệ thống tự động sử dụng model.livr_stage để sinh ra mask tương ứng.
        outputs = model(**inputs)
        
        # Lấy giá trị Loss của lượt chạy này và chia cho các bước tích lũy
        loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward() # Lan truyền ngược ghi đạo hàm vào giấy nháp
        
        epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        
        # --- THỰC THI TÍCH LŨY GRADIENT CHỐNG TRÀN VRAM CHUẨN Ý THẦY ---
        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(cleaned_dataset):
            # Cắt bớt gradient nếu quá lớn để chống bùng nổ đạo hàm (Gradient Clipping)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            
            optimizer.step()      # Cập nhật bộ não dựa trên giấy nháp tổng hợp từ 8 ảnh
            optimizer.zero_grad()  # Xóa sạch giấy nháp để chuẩn bị cho chu kỳ 8 ảnh tiếp theo
            
        # Cập nhật thông số Loss liên tục lên màn hình console
        progress_bar.set_postfix({"Loss": f"{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}"})
        
    print(f"➔ Kết thúc Epoch {epoch} - Average Loss tổng thể: {epoch_loss / len(cleaned_dataset):.4f}")

# --- BƯỚC CUỐI: ĐÓNG GÓI VÀ LƯU TRỰC TIẾP CHECKPOINT RA GOOGLE DRIVE ---
# Tạo thư mục checkpoints nếu chưa có
drive_checkpoint_dir = "/content/drive/MyDrive/LIVR_Mini_Project/checkpoints"
os.makedirs(drive_checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(drive_checkpoint_dir, "livr_mini_checkpoint.pt")

# Lưu các tham số LoRA weights và Latent Embeddings (các tham số requires_grad=True)
torch.save({
    'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items() if v.requires_grad},
    'latent_embeddings': model.base_model.model.model.embed_tokens.weight[manager.latent_token_ids].detach().cpu(),
    'latent_token_ids': manager.latent_token_ids
}, checkpoint_path)

print(f"\n[SUCCESS] Hoàn thành trọn vẹn phần Implement (Mini)!")
print(f"File trọng số thông minh đã được lưu an toàn tại: {checkpoint_path}")